In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Logistic Regression vs. K-Means + Logistic Regression

이 노트북은 `diabetic_data.csv`를 사용하여 당뇨 환자의 **30일 이내 재입원 여부**를 예측합니다.

- Target: `readmitted`
- Positive class: `readmitted == '<30'`이면 1
- Negative class: `readmitted == 'NO'` 또는 `readmitted == '>30'`이면 0
- 비교 모델 1: Logistic Regression only
- 비교 모델 2: K-Means cluster label을 feature로 추가한 Logistic Regression

K-Means는 단독 예측 모델로 사용하지 않고, train set에서만 fit한 뒤 cluster label을 생성하는 feature engineering 단계로 사용합니다.

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    silhouette_score
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH = '/content/drive/MyDrive/SKKU/M2_1/diabetic_data.csv'
RESULT_PATH = 'logistic_regression_results.csv'

sns.set_theme(style='whitegrid')

## 2. Load Dataset

In [ ]:
# Google Drive에 저장된 CSV 파일을 불러옵니다.
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
display(df.head())
display(df.info())

## 3. Define Target Variable

In [ ]:
# readmitted 값 중 '<30'만 30일 이내 재입원으로 정의합니다.
# 'NO'와 '>30'은 30일 이내 재입원이 아니므로 0으로 변환합니다.
df['target_30day_readmission'] = (df['readmitted'] == '<30').astype(int)

print('Original readmitted distribution')
display(df['readmitted'].value_counts(dropna=False))

print('\nBinary target distribution')
display(df['target_30day_readmission'].value_counts(normalize=True).rename('ratio'))

plt.figure(figsize=(5, 4))
sns.countplot(data=df, x='target_30day_readmission')
plt.title('30-Day Readmission Target Distribution')
plt.xlabel('0 = No 30-day readmission, 1 = <30 days')
plt.ylabel('Count')
plt.show()

## 4. Basic Data Cleaning

In [ ]:
# '?'는 이 데이터에서 결측값을 의미하므로 NaN으로 바꿉니다.
data = df.replace('?', np.nan).copy()

# 식별자 변수와 결측 비율이 매우 높은 변수는 제거합니다.
# readmitted 원본 변수도 target으로 변환했으므로 feature에서는 제거합니다.
drop_columns = [
    'encounter_id',
    'patient_nbr',
    'weight',
    'payer_code',
    'medical_specialty',
    'readmitted'
]

data = data.drop(columns=drop_columns)

# age는 '[50-60)' 같은 구간 문자열이므로 중앙값 숫자로 변환합니다.
def age_to_midpoint(age_value):
    if pd.isna(age_value):
        return np.nan
    left, right = age_value.strip('[]()').split('-')
    return (int(left) + int(right)) / 2

# ICD 진단 코드는 숫자의 크기 자체보다 질병군 정보가 중요하므로 큰 범주로 묶습니다.
def diagnosis_group(code):
    if pd.isna(code):
        return 'Missing'
    try:
        value = float(code)
    except ValueError:
        return 'Other'

    if 390 <= value <= 459 or value == 785:
        return 'Circulatory'
    if 460 <= value <= 519 or value == 786:
        return 'Respiratory'
    if 520 <= value <= 579 or value == 787:
        return 'Digestive'
    if 250 <= value < 251:
        return 'Diabetes'
    if 800 <= value <= 999:
        return 'Injury'
    if 710 <= value <= 739:
        return 'Musculoskeletal'
    if 580 <= value <= 629 or value == 788:
        return 'Genitourinary'
    if 140 <= value <= 239:
        return 'Neoplasms'
    return 'Other'

data['age_midpoint'] = data['age'].apply(age_to_midpoint)
data = data.drop(columns=['age'])

for col in ['diag_1', 'diag_2', 'diag_3']:
    data[f'{col}_group'] = data[col].apply(diagnosis_group)
data = data.drop(columns=['diag_1', 'diag_2', 'diag_3'])

# admission/discharge/source ID는 숫자처럼 보이지만 범주형 코드입니다.
id_like_categorical_columns = ['admission_type_id', 'discharge_disposition_id', 'admission_source_id']
for col in id_like_categorical_columns:
    data[col] = data[col].astype('object')

print('Cleaned data shape:', data.shape)
display(data.head())

missing_summary = data.isna().mean().sort_values(ascending=False).head(15)
display((missing_summary * 100).round(2).to_frame('missing_percent'))

## 5. Train/Test Split

In [ ]:
# target과 feature를 분리합니다.
X = data.drop(columns=['target_30day_readmission'])
y = data['target_30day_readmission']

# 두 모델이 반드시 같은 train/test split을 사용하도록 여기서 한 번만 split합니다.
# stratify=y를 사용하여 30일 이내 재입원 비율이 train/test에 비슷하게 유지되도록 합니다.
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('X_train:', X_train.shape)
print('X_test:', X_test.shape)
print('\nTrain target ratio')
display(y_train.value_counts(normalize=True).rename('ratio'))
print('\nTest target ratio')
display(y_test.value_counts(normalize=True).rename('ratio'))

## 6. Preprocessing

In [ ]:
# train/test split 이후에만 전처리기를 fit합니다.
# 이렇게 해야 test set의 정보가 train 단계로 새어 들어가는 데이터 누수를 막을 수 있습니다.
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='Missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Logistic Regression only와 K-Means 모두 동일하게 scaling/encoding된 행렬을 사용합니다.
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

feature_names = preprocessor.get_feature_names_out()

print('Number of numeric features:', len(numeric_features))
print('Number of categorical features:', len(categorical_features))
print('Preprocessed train shape:', X_train_preprocessed.shape)
print('Preprocessed test shape:', X_test_preprocessed.shape)

## 7. Baseline Model: Logistic Regression

In [ ]:
# Baseline 모델은 K-Means cluster label 없이 전처리된 feature만 사용합니다.
baseline_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='liblinear',
    random_state=RANDOM_STATE
)

baseline_model.fit(X_train_preprocessed, y_train)

baseline_pred = baseline_model.predict(X_test_preprocessed)
baseline_proba = baseline_model.predict_proba(X_test_preprocessed)[:, 1]

print('Baseline Logistic Regression training complete.')

## 8. K-Means Clustering

In [ ]:
# K-Means도 동일하게 전처리된 train matrix를 사용합니다.
# k=3과 k=4를 중심으로 확인하고, 참고용으로 elbow/silhouette score를 계산합니다.
k_values = [2, 3, 4, 5, 6]
kmeans_diagnostics = []

# silhouette score는 전체 데이터에서 계산하면 오래 걸릴 수 있어 train set 일부 샘플만 사용합니다.
sample_size = min(10000, X_train_preprocessed.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
sample_indices = rng.choice(X_train_preprocessed.shape[0], size=sample_size, replace=False)
X_train_sample = X_train_preprocessed[sample_indices]

for k in k_values:
    temp_kmeans = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    temp_labels = temp_kmeans.fit_predict(X_train_preprocessed)
    sample_labels = temp_labels[sample_indices]
    score = silhouette_score(X_train_sample, sample_labels)
    kmeans_diagnostics.append({
        'k': k,
        'inertia': temp_kmeans.inertia_,
        'silhouette_score': score
    })

kmeans_diagnostics_df = pd.DataFrame(kmeans_diagnostics)
display(kmeans_diagnostics_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=kmeans_diagnostics_df, x='k', y='inertia', marker='o', ax=axes[0])
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of clusters k')
axes[0].set_ylabel('Inertia')

sns.lineplot(data=kmeans_diagnostics_df, x='k', y='silhouette_score', marker='o', ax=axes[1])
axes[1].set_title('Silhouette Score')
axes[1].set_xlabel('Number of clusters k')
axes[1].set_ylabel('Silhouette score')
plt.tight_layout()
plt.show()

# 수업 프로젝트에서 설명하기 쉽도록 최종 모델은 k=4를 사용합니다.
# 위 표/그래프를 보고 k=3이 더 적절하다고 판단되면 SELECTED_K를 3으로 바꾸면 됩니다.
SELECTED_K = 4
print('Selected k for final model:', SELECTED_K)

In [ ]:
# 데이터 누수를 막기 위해 K-Means는 train set에만 fit합니다.
# test set에는 fit하지 않고 predict만 적용합니다.
kmeans = KMeans(n_clusters=SELECTED_K, random_state=RANDOM_STATE, n_init=10)
train_clusters = kmeans.fit_predict(X_train_preprocessed)
test_clusters = kmeans.predict(X_test_preprocessed)

cluster_profile = pd.DataFrame({
    'cluster': train_clusters,
    'target_30day_readmission': y_train.values
})

cluster_summary = cluster_profile.groupby('cluster').agg(
    count=('target_30day_readmission', 'size'),
    readmission_rate=('target_30day_readmission', 'mean')
).sort_values('readmission_rate', ascending=False)

display(cluster_summary)

plt.figure(figsize=(7, 4))
sns.barplot(data=cluster_summary.reset_index(), x='cluster', y='readmission_rate')
plt.title('30-Day Readmission Rate by K-Means Cluster')
plt.xlabel('Cluster')
plt.ylabel('30-day readmission rate')
plt.show()

## 9. Model: K-Means + Logistic Regression

In [ ]:
# cluster label은 범주형 변수처럼 one-hot encoding하여 Logistic Regression에 추가합니다.
# 두 모델의 차이는 이 cluster feature가 추가되었는지 여부뿐입니다.
cluster_encoder = OneHotEncoder(handle_unknown='ignore')
train_cluster_encoded = cluster_encoder.fit_transform(train_clusters.reshape(-1, 1))
test_cluster_encoded = cluster_encoder.transform(test_clusters.reshape(-1, 1))

from scipy.sparse import hstack

X_train_with_cluster = hstack([X_train_preprocessed, train_cluster_encoded])
X_test_with_cluster = hstack([X_test_preprocessed, test_cluster_encoded])

cluster_feature_names = cluster_encoder.get_feature_names_out(['kmeans_cluster'])
feature_names_with_cluster = np.concatenate([feature_names, cluster_feature_names])

kmeans_lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',
    solver='liblinear',
    random_state=RANDOM_STATE
)

kmeans_lr_model.fit(X_train_with_cluster, y_train)

kmeans_lr_pred = kmeans_lr_model.predict(X_test_with_cluster)
kmeans_lr_proba = kmeans_lr_model.predict_proba(X_test_with_cluster)[:, 1]

print('K-Means + Logistic Regression training complete.')
print('Train matrix with cluster:', X_train_with_cluster.shape)
print('Test matrix with cluster:', X_test_with_cluster.shape)

## 10. Model Evaluation

In [ ]:
def evaluate_model(model_name, y_true, y_pred, y_proba):
    """주요 binary classification 평가 지표를 계산하고 출력합니다."""
    metrics = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_true, y_proba)
    }

    print('=' * 80)
    print(model_name)
    print('=' * 80)
    print('Accuracy :', round(metrics['Accuracy'], 4))
    print('Precision:', round(metrics['Precision'], 4))
    print('Recall   :', round(metrics['Recall'], 4))
    print('F1-score :', round(metrics['F1-score'], 4))
    print('ROC-AUC  :', round(metrics['ROC-AUC'], 4))

    print('\nConfusion Matrix')
    cm = confusion_matrix(y_true, y_pred)
    display(pd.DataFrame(
        cm,
        index=['Actual 0', 'Actual 1'],
        columns=['Predicted 0', 'Predicted 1']
    ))

    print('\nClassification Report')
    print(classification_report(
        y_true,
        y_pred,
        target_names=['No 30-day readmission', '30-day readmission'],
        zero_division=0
    ))

    return metrics

baseline_metrics = evaluate_model(
    'Logistic Regression only',
    y_test,
    baseline_pred,
    baseline_proba
)

kmeans_lr_metrics = evaluate_model(
    'K-Means + Logistic Regression',
    y_test,
    kmeans_lr_pred,
    kmeans_lr_proba
)

## 11. Performance Comparison

In [ ]:
# 두 모델의 성능을 하나의 DataFrame으로 정리합니다.
results_df = pd.DataFrame([baseline_metrics, kmeans_lr_metrics])
metric_columns = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'ROC-AUC']
results_df[metric_columns] = results_df[metric_columns].round(4)

display(results_df)

# 결과 비교표를 CSV 파일로 저장합니다.
results_df.to_csv(RESULT_PATH, index=False)
print(f'Saved results to {RESULT_PATH}')

results_long = results_df.melt(id_vars='Model', value_vars=metric_columns, var_name='Metric', value_name='Score')

plt.figure(figsize=(10, 5))
sns.barplot(data=results_long, x='Metric', y='Score', hue='Model')
plt.title('Model Performance Comparison')
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.show()

## 12. Interpretation

In [ ]:
def show_top_coefficients(model, names, title, top_n=15):
    """Logistic Regression coefficient 절댓값이 큰 feature를 확인합니다."""
    coef_df = pd.DataFrame({
        'feature': names,
        'coefficient': model.coef_[0]
    })
    coef_df['abs_coefficient'] = coef_df['coefficient'].abs()
    top_coef = coef_df.sort_values('abs_coefficient', ascending=False).head(top_n)

    print(title)
    display(top_coef[['feature', 'coefficient']])

    plt.figure(figsize=(9, 6))
    plot_data = top_coef.sort_values('coefficient')
    sns.barplot(data=plot_data, x='coefficient', y='feature', palette='vlag')
    plt.axvline(0, color='black', linewidth=1)
    plt.title(title)
    plt.xlabel('Coefficient')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.show()

    return coef_df

baseline_coef_df = show_top_coefficients(
    baseline_model,
    feature_names,
    'Top 15 Coefficients: Logistic Regression only'
)

kmeans_lr_coef_df = show_top_coefficients(
    kmeans_lr_model,
    feature_names_with_cluster,
    'Top 15 Coefficients: K-Means + Logistic Regression'
)

### Interpretation Notes

- 이 문제는 의료 재입원 예측이므로 Accuracy만 보면 안 됩니다. 실제 30일 이내 재입원 환자를 놓치지 않는 것이 중요하므로 **Recall**, **F1-score**, **ROC-AUC**를 함께 비교해야 합니다.
- `Logistic Regression only`는 baseline입니다.
- `K-Means + Logistic Regression`은 같은 전처리 feature에 K-Means cluster label만 추가한 모델입니다.
- 두 모델의 성능 차이가 크지 않더라도, cluster label의 coefficient와 cluster별 재입원율을 보면 K-Means가 환자군 차이를 어느 정도 포착했는지 해석할 수 있습니다.
- K-Means는 train set에만 fit했고 test set에는 predict만 적용했으므로, test set 정보가 학습 단계에 들어가는 데이터 누수를 피했습니다.